# Bin Buddy · WS1 · Notebook 04 — Final model: calibration, embedding, export, versioning

Turns the chosen model (notebook 02) into the files the model API loads:

1. **Temperature scaling** — makes "90% sure" mean right ~90% of the time, because the app's
   confidence gate (ask questions below 0.80) relies on it.
2. **A 256-value embedding** (contract `EMBEDDING_DIM = 256`, L2-normalised) for disposal
   verification (WS4) and out-of-distribution checks (WS2): the pooled CNN features reduced with
   PCA. PCA is linear, so it is exported as a matrix, and it keeps the directions of largest
   variance.
3. **ONNX export** (runs on CPU with ONNX Runtime, no PyTorch needed in the app) plus the
   classifier weights (and ConvNeXt's LayerNorm) for the NumPy Grad-CAM in notebook 03, section 5.
   Then I run the **API's own code** (`services/inference/app/vision.py`) on the exported files and
   compare it with PyTorch, so what I upload is exactly what the app will compute.
4. **Versioning on the Hugging Face Hub** with a tag (`v0.2-local-data` for this model), so the API
   loads a pinned revision and every result traces to one model version.

## 0. Setup

I run this on **Google Colab**: a **T4 GPU** for notebooks 02 and 03 (training and Score-CAM need
it); a CPU is enough for 01 and 04 (they only prepare data or run the trained model forward). The cell
below mounts Google Drive, clones the team repository (branch `ws1/vision-model`) and installs pinned
package versions, so every run uses the same code and versions.

`SMOKE=1` is only for my local test run on tiny synthetic images (a "smoke test": it checks that every
cell runs end to end in minutes, before I spend GPU time); leave it unset on Colab.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

SMOKE = os.environ.get("BINBUDDY_SMOKE") == "1"  # my local test on synthetic data only

# Experiment tracking: every data version and every run keeps its own files, so I never
# overwrite an earlier result. Before a new run I add it to ml/binbuddy_ml/experiments.py
# (RUNS) and to ml/EXPERIMENTS.md, then set the two values below.
#   Data v1 = public data + 207 team photos (14 Oct)        -> Run 1
#   Data v2 = v1 + the final team photo batch (frozen)      -> Runs 2 and 3
#   Run 2 = Run 1 settings on v2 (effect of the photos); Run 3 = anti-overfitting settings on v2
DATA_VERSION = os.environ.get("BINBUDDY_DATA_VERSION", "v2")
# Default = the run whose model I chose for the app (Run 3, ConvNeXt-T), so notebooks 03 and 04
# explain and export exactly that model. Notebook 02 skips finished models if re-run.
RUN_ID = os.environ.get("BINBUDDY_RUN_ID", "run3")
REPO_URL = "https://github.com/DevelopingGod/Group-5-PRS-Practice-Module-2026.git"
BRANCH = "ws1/vision-model"
# I pin package versions so a rerun next week uses the same code as today.
PINNED = ["timm==1.0.30", "grad-cam==1.5.7", "open-clip-torch==3.3.0", "imagehash==4.3.2"]

if SMOKE:
    REPO = Path(os.environ["BINBUDDY_REPO"])
    DATA = Path(os.environ["BINBUDDY_DATA_ROOT"])
    WORK = DATA / "_work"
else:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO = Path("/content/repo")
    DATA = Path("/content/drive/MyDrive/BinBuddy_Data")
    WORK = Path("/content/work")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINNED], check=True)

# The contracts and helpers come straight from the repository (no copy-paste drift).
for p in (REPO / "contracts", REPO / "ml", REPO / "services" / "inference"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
WORK.mkdir(parents=True, exist_ok=True)
PREP = WORK / f"prepared_{DATA_VERSION}"  # resized images, extracted from TAR
TAR = DATA / f"prepared_{DATA_VERSION}.tar"
# Run 1 / data v1 used the original, unversioned names; later versions get a suffix.
MANIFEST = (
    DATA
    / "manifests"
    / ("vision_manifest.csv" if DATA_VERSION == "v1" else f"vision_manifest_{DATA_VERSION}.csv")
)
RESULTS = DATA / "results" if RUN_ID == "run1" else DATA / "results" / RUN_ID
MODELS_DIR = DATA / "models" if RUN_ID == "run1" else DATA / "models" / RUN_ID
for d in (MANIFEST.parent, RESULTS, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)
# The code version is printed and saved with every result, so each number can be traced to code.
COMMIT = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True
).stdout.strip()
print(f"run {RUN_ID} | data {DATA_VERSION} | code {COMMIT} | smoke {SMOKE}")
print("results ->", RESULTS)

In [ ]:
import tarfile

import pandas as pd

if not PREP.exists():
    with tarfile.open(TAR) as tf:  # one big file from Drive is much faster than 10k small ones
        tf.extractall(PREP, filter="data")
df = pd.read_csv(MANIFEST)
print(len(df), "images")
print(df.groupby(["source", "split"]).size().unstack(fill_value=0))

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms as T

from binbuddy_contracts.enums import Material
from binbuddy_ml import experiments as ex

CLASSES = [m.value for m in Material]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PRETRAINED = os.environ.get("BINBUDDY_SMOKE_PRETRAINED", "1") == "1"
df["label"] = df["material"].map(CLASSES.index)
# I read the model chosen in notebook 02 (for RUN_ID); Run 3 chose ConvNeXt-T.
chosen = json.loads((RESULTS / "chosen_model.json").read_text())
EXP = chosen["experiment"]
FINE_TUNED = [m.key for m in ex.MODELS if m.mode is ex.Mode.FINE_TUNE]
# Grad-CAM needs a network trained end to end, so a probe/zero-shot winner is explained
# through the best fine-tuned model instead (and the report says so).
res = (
    pd.DataFrame(json.loads((RESULTS / "comparison_results.json").read_text()))
    if (RESULTS / "comparison_results.json").exists()
    else None
)
if chosen["model"] in FINE_TUNED:
    KEY = chosen["model"]
elif res is not None:
    ft = res[(res["experiment"] == EXP) & (res["mode"] == "fine_tune")]
    KEY = ft.sort_values("macro_f1", ascending=False).iloc[0]["model"]
else:
    KEY = "efficientnet_b0"
print("explaining:", KEY, "| experiment:", EXP, "| run:", RUN_ID)


def load(key):
    # Dropout / drop path only act during training, so I rebuild the plain architecture and load
    # the trained weights into it; the parameter shapes are identical.
    spec = ex.model(key)
    m = timm.create_model(spec.timm_id, pretrained=False, num_classes=len(CLASSES))
    weights = MODELS_DIR / f"{EXP}_{key}.pt"
    if weights.exists():
        m.load_state_dict(torch.load(weights, map_location="cpu"))
    else:
        print("WARNING: no fine-tuned weights for", key, "- using untrained head")
    return m.to(DEVICE).eval()


def eval_transform(model):
    cfg = timm.data.resolve_data_config({}, model=model)
    tf = T.Compose(
        [T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(cfg["mean"], cfg["std"])]
    )
    show = T.Compose([T.Resize(256), T.CenterCrop(224)])
    return tf, show


def vit_tokens_to_map(t):
    # DeiT / CLIP ViT: drop the class token, lay the 196 patch tokens back out as a 14 x 14 grid.
    t = t[:, 1:, :]
    side = int(t.shape[1] ** 0.5)
    return t.reshape(t.shape[0], side, side, t.shape[2]).permute(0, 3, 1, 2)


def target_layer(key, m):
    """Last spatial feature map before pooling, per architecture."""
    if key == "efficientnet_b0":
        return m.bn2, None  # conv_head -> BN -> SiLU output (1280 x 7 x 7)
    if key == "mobilenetv3_l":
        return m.blocks[-1], None  # MobileNetV3 pools *before* its conv_head
    if key == "resnet50":
        return m.layer4[-1], None
    if key == "convnext_t":
        return m.stages[-1], None
    if key == "swin_t":
        return m.layers[-1], lambda t: t.permute(0, 3, 1, 2)  # timm Swin is channels-last
    if key in {"deit_s", "clip_b16_ft"}:
        return m.blocks[-1].norm1, vit_tokens_to_map
    raise KeyError(key)


# I explain held-out *team* test photos: they look like what users will send, and the model never
# saw them in training.
test = df[(df["split"] == "test") & (df["source"] == "local")]
if not len(test):
    test = df[df["split"] == "test"]
N_EVAL = 4 if SMOKE else 40
eval_rows = test.sample(min(N_EVAL, len(test)), random_state=0)

In [ ]:
if not SMOKE:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "onnx==1.23.2", "onnxruntime==1.31.0"],
        check=True,
    )
from sklearn.decomposition import PCA

from binbuddy_contracts.constants import EMBEDDING_DIM

HF_REPO = "sankalp-indish/bin-buddy-vision"  # create once on huggingface.co (free)
# The tag is what the API pins (MODEL_REVISION). I never move a tag: a new model gets a new tag.
TAG = (
    "v0.2-local-data" if EXP.startswith("E2") else "v0.1-baseline"
)  # trained with / without team photos
OUT = WORK / "export"
OUT.mkdir(exist_ok=True)
model = load(KEY).float()
tf, _ = eval_transform(model)
spec = ex.model(KEY)
# I fit the temperature on *team* validation photos: the confidence gate must be honest on the
# photos users actually send, and these were never used for training or testing.
local_val = df[(df["split"] == "val") & (df["source"] == "local")]
calib = local_val if len(local_val) >= 20 else df[df["split"] == "val"]
print("model:", KEY, "| calibration images:", len(calib))

## 1. Temperature scaling (Guo et al., 2017)
One number $T > 0$ divides the logits before softmax: $p = \mathrm{softmax}(z / T)$. $T$ is fitted
by minimising the negative log-likelihood on validation images. It never changes *which* class
wins (accuracy is identical), only how confident the model claims to be.

In [ ]:
import itertools


@torch.no_grad()
def logits_and_pooled(frame, max_n=None):
    zs, ps, ys = [], [], []
    for i, (_, r) in enumerate(frame.iterrows()):
        if max_n and i >= max_n:
            break
        x = tf(Image.open(PREP / r["path"]).convert("RGB"))[None].to(DEVICE)
        f = model.forward_features(x)
        pooled = model.forward_head(f, pre_logits=True)
        zs.append(model.get_classifier()(pooled).float().cpu())
        ps.append(pooled.float().cpu())
        ys.append(r["label"])
    return torch.cat(zs), torch.cat(ps), torch.tensor(ys)


z, _, y = logits_and_pooled(calib, 16 if SMOKE else None)
# I optimise log T rather than T so T always stays positive.
log_t = torch.zeros(1, requires_grad=True)
opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)


def closure():
    opt.zero_grad()
    loss = F.cross_entropy(z / log_t.exp(), y)
    loss.backward()
    return loss


opt.step(closure)
TEMPERATURE = float(log_t.exp())


def ece(p, y, bins=15):
    conf, hit = p.max(1).values, (p.argmax(1) == y).float()
    total = 0.0
    for lo, hi in itertools.pairwise(torch.linspace(0, 1, bins + 1)):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += float(m.float().mean() * (hit[m].mean() - conf[m].mean()).abs())
    return total


print(
    f"T = {TEMPERATURE:.3f} | ECE before {ece(z.softmax(1), y):.3f} -> after {ece((z / TEMPERATURE).softmax(1), y):.3f}"
)

## 2. The 256-value embedding
Pooled features (1280 values for EfficientNet-B0) → PCA fitted on training images → 256 values →
L2-normalised, so cosine similarity is a dot product. The check below shows whether two photos of
the *same local item* are more similar than photos of different items — the property disposal
verification needs (contract `VERIFY_MIN_SIMILARITY = 0.85`, PROVISIONAL, owned by WS4).

In [ ]:
# PCA is fitted on training images only (4,000 is plenty for 768 -> 256 dimensions).
train = df[df["split"] == "train"].sample(min(4000, (df["split"] == "train").sum()), random_state=0)
_, feats, _ = logits_and_pooled(train, 64 if SMOKE else None)
n_comp = min(EMBEDDING_DIM, feats.shape[0], feats.shape[1])
pca = PCA(n_components=n_comp, random_state=0).fit(feats.numpy())
print(
    f"PCA {feats.shape[1]} -> {n_comp}: keeps {pca.explained_variance_ratio_.sum():.1%} of the variance"
)
assert n_comp == EMBEDDING_DIM or SMOKE, (
    "need >= 256 training images and features for a 256-d embedding"
)


def embed(pooled):
    e = pca.transform(pooled.numpy())
    return e / np.linalg.norm(e, axis=1, keepdims=True)


loc = df[df["source"] == "local"]
if loc["group"].duplicated().any():
    _, lp, _ = logits_and_pooled(loc, 40 if SMOKE else None)
    sub = loc.head(len(lp))
    e = embed(lp)
    sim = e @ e.T
    same = sub["group"].to_numpy()[:, None] == sub["group"].to_numpy()[None, :]
    iu = np.triu_indices(len(sub), 1)
    s_same, s_diff = sim[iu][same[iu]], sim[iu][~same[iu]]
    plt.hist(s_diff, bins=40, alpha=0.6, density=True, label="different items")
    plt.hist(s_same, bins=40, alpha=0.6, density=True, label="same item")
    plt.axvline(0.85, color="k", ls="--", label="VERIFY_MIN_SIMILARITY 0.85")
    plt.xlabel("cosine similarity")
    plt.legend()
    plt.show()
    print(
        f"same item: median {np.median(s_same):.2f} | different items: 95th pct {np.percentile(s_diff, 95):.2f}"
    )

## 3. Export: ONNX graph + head + PCA, verified against PyTorch
The ONNX graph outputs the logits, the pooled features and the last feature map. The API computes
probabilities (with $T$), the embedding (PCA matrix) and the Grad-CAM heatmap (classifier weights
× feature map) in NumPy.

In [ ]:
import onnxruntime as ort


class Exported(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        fmap = self.m.forward_features(x)
        pooled = self.m.forward_head(fmap, pre_logits=True)
        return self.m.get_classifier()(pooled), pooled, fmap


wrapped = Exported(model.cpu()).eval()  # the exporter restores this mode afterwards
x = torch.randn(1, 3, 224, 224)
onnx_path = OUT / "model.onnx"
torch.onnx.export(
    wrapped,
    x,
    onnx_path,
    input_names=["image"],
    output_names=["logits", "pooled", "feature_map"],
    dynamic_axes={"image": {0: "batch"}},
    opset_version=17,
    dynamo=False,
)

sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
with torch.no_grad():
    ref = [t.numpy() for t in wrapped(x)]
got = sess.run(None, {"image": x.numpy()})
diff = max(
    float(np.abs(a - b).max() / max(np.abs(a).max(), 1e-6)) for a, b in zip(ref, got, strict=True)
)
print(f"ONNX vs PyTorch: max relative difference {diff:.2e}")
assert diff < 1e-4

cfg = timm.data.resolve_data_config({}, model=model)
head = model.get_classifier()
arrays = {
    "classifier_weight": head.weight.detach().numpy(),
    "classifier_bias": head.bias.detach().numpy(),
    "pca_components": pca.components_,
    "pca_mean": pca.mean_,
}
# ConvNeXt: pooled features pass a LayerNorm before the classifier. The API needs its weight to
# compute the exact Grad-CAM channel weights (notebook 03, section 5).
norm = getattr(getattr(model, "head", None), "norm", None)
if isinstance(norm, torch.nn.LayerNorm):
    arrays["norm_weight"] = norm.weight.detach().numpy()
    arrays["norm_bias"] = norm.bias.detach().numpy()
np.savez(OUT / "head.npz", **arrays)
meta = {
    "model_key": KEY,
    "timm_id": spec.timm_id,
    "classes": CLASSES,
    "input_size": 224,
    "resize": 256,
    "mean": list(cfg["mean"]),
    "std": list(cfg["std"]),
    "temperature": TEMPERATURE,
    "embedding_dim": int(n_comp),
    "head_norm_eps": float(norm.eps) if "norm_weight" in arrays else None,
    "experiment": EXP,
    "tag": TAG,
    "run": RUN_ID,
    "data_version": DATA_VERSION,
    "code_commit": COMMIT,
}
(OUT / "config.json").write_text(json.dumps(meta, indent=2))
torch.save(model.state_dict(), OUT / "pytorch_weights.pt")
sorted(p.name for p in OUT.iterdir())

## 3b. End-to-end check with the API's own code
I load the exported folder with `VisionModel` from `services/inference/app/vision.py` (exactly what the
deployed API runs) and compare its probabilities and heatmap with PyTorch on team test photos. If this
passes, the app and the report show the same model.

In [ ]:
from app import vision

if SMOKE and n_comp != EMBEDDING_DIM:
    # My tiny synthetic smoke data cannot fill a 256-d PCA, and the API refuses any other size.
    print("SMOKE: API check skipped (embedding", n_comp, "!= contract", EMBEDDING_DIM, ")")
    check_rows = df.head(0)
else:
    api_model = vision.VisionModel.load(OUT)
    check_rows = df[(df["split"] == "test") & (df["source"] == "local")].head(4 if SMOKE else 12)
model.cpu().eval()
prob_diff, heat_r = [], []
for path in check_rows["path"]:
    img = Image.open(PREP / path).convert("RGB")
    res = api_model.analyse(img)
    api_p = np.array([s.probability for s in res.material_scores])
    x = tf(img)[None]
    with torch.no_grad():
        ref_p = (model(x) / TEMPERATURE).softmax(1)[0].numpy()
    prob_diff.append(float(np.abs(api_p - ref_p).max()))
    # Heatmap: Grad-CAM by its definition (autograd on the last feature map, same input as the API)
    # vs the API's NumPy formula, both at the feature-map resolution (7 x 7), before colouring.
    xa, _ = api_model.preprocess(img)
    A = model.forward_features(torch.from_numpy(xa.copy()))
    A.retain_grad()
    logits = model.forward_head(A)
    top = int(logits.argmax())
    logits[0, top].backward()
    ref = torch.relu((A.grad.mean((2, 3))[0][:, None, None] * A[0]).sum(0)).detach().numpy()
    ref = (ref - ref.min()) / max(float(ref.max() - ref.min()), 1e-12)
    f = A[0].detach().numpy()
    grad = vision.head_gradient(
        f.mean(axis=(1, 2)),
        api_model.classifier_weight[top],
        api_model.norm_weight,
        api_model.norm_eps,
    )
    heat_r.append(
        float(np.corrcoef(vision.class_activation_map(f, grad).ravel(), ref.ravel())[0, 1])
    )
if prob_diff:
    print(
        f"API vs PyTorch: max probability difference {max(prob_diff):.4f} | heatmap Pearson r (mean) {np.nanmean(heat_r):.3f}"
    )
    # Small probability differences come from PIL vs torchvision resizing; the class must not change.
    assert max(prob_diff) < 0.05, "the API computes something different from the trained model"
    assert np.nanmin(heat_r) > 0.999, "the API heatmap is not Grad-CAM"

## 4. Model card and upload to the Hugging Face Hub
Needs a free Hugging Face account, a model repository named in `HF_REPO`, and a **write** token
stored in Colab (key icon on the left → Secrets → `HF_TOKEN`, notebook access on). The token is
never printed or saved in the notebook.

In [ ]:
results_file = RESULTS / "comparison_results.json"
# The card quotes this run's own test results for the exported model (never numbers typed by hand).
rows = (
    [r for r in json.loads(results_file.read_text()) if r["model"] == KEY]
    if results_file.exists()
    else []
)
lines = [
    f"| {r['experiment']} | {r['test_set']} | {r['macro_f1']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}] | "
    f"{r['accuracy']:.3f} | {r['costly_error']:.3f} | {r['ece']:.3f} |"
    for r in rows
]
card = f"""---
license: other
tags: [image-classification, recycling, grad-cam, binbuddy]
---
# Bin Buddy vision model ({TAG})

Predicts the **material** of one household item in a phone photo (10 classes: {", ".join(CLASSES)}).
Cleanliness is *not* predicted; the Bin Buddy app asks about it. Advisory only; defers to NEA guidance.
NUS-ISS PRS Practice Module, Group 5, workstream WS1.

- Backbone: `{spec.timm_id}` ({spec.family}), fine-tuned in {RUN_ID} (data {DATA_VERSION}, code {COMMIT}); temperature {TEMPERATURE:.3f}
- Embedding: {n_comp}-d PCA of pooled features, L2-normalised
- Training data: TrashNet (MIT), RealWaste (CC BY-NC-SA 4.0: **non-commercial use only**), TACO (CC BY 4.0), team photos

| Experiment | Test set | Macro-F1 [95% CI] | Accuracy | Costly error | ECE |
|---|---|---|---|---|---|
{chr(10).join(lines)}
"""
(OUT / "README.md").write_text(card)
print(card)

In [ ]:
if not SMOKE:
    from google.colab import userdata
    from huggingface_hub import HfApi

    # The write token comes from Colab Secrets; it is never printed or stored in the notebook.
    api = HfApi(token=userdata.get("HF_TOKEN"))
    api.create_repo(
        HF_REPO, repo_type="model", exist_ok=True, private=True
    )  # RealWaste is non-commercial (CC BY-NC-SA)
    commit = api.upload_folder(
        repo_id=HF_REPO, folder_path=str(OUT), commit_message=f"{TAG}: {KEY}"
    )
    api.create_tag(HF_REPO, tag=TAG, revision=commit.oid, exist_ok=False)
    print("uploaded:", f"https://huggingface.co/{HF_REPO}/tree/{TAG}")